# Exercise 1

In this exericise, we implement a function `gamble(m)` that simulates the gambling game from the lectures. The function `gamble(m)` receives an integer `m` for the initial amount of money, and in each round randomly changes the money to `m+1` or `m-1` until the money becomes 0, and then returns the total number of rounds that the gambler loses all their money. (From the lecture we know that with probability $1$ the function `gamble(m)` will terminate for any input `m`.)

In [ ]:
import numpy as np

def gamble(m):
    # Number of rounds until m = 0
    rounds = 0

    # Iterate until m = 0
    while m > 0:
        # Randomly decide whether to increment or decrement m
        random_choice = np.random.randint(2)

        # Update m with the random increment or decrement
        m += ((random_choice * 2) - 1)

        # Increment the number of rounds
        rounds += 1

    return rounds

Run `gamble(1)` 100 times and compute the average number of rounds that the gamble plays until they lose all the money.

In [19]:
# Initialise a variable to hold the sum of the number of rounds
sum_of_rounds = 0

# Call the gamble function 100 times
for i in range(100):
    sum_of_rounds += gamble(1)

# Calculate the mean number of rounds
mean_rounds = sum_of_rounds / 100

# Output the average number of rounds
print(f"Average number of rounds for 100 trials: {mean_rounds}")

Average number of rounds for 100 trials: 3677.76


Run `gamble(1)` 1000 times and compute the average number of rounds that the gamble plays until they lose all the money.

In [20]:
# Initialise a variable to hold the sum of the number of rounds
sum_of_rounds = 0

# Call the gamble function 100 times
for i in range(1000):
    sum_of_rounds += gamble(1)

# Calculate the mean number of rounds
mean_rounds = sum_of_rounds / 1000

# Output the average number of rounds
print(f"Average number of rounds for 1000 trials: {mean_rounds}")

Average number of rounds for 1000 trials: 8366.348


Run `gamble(1)` 10000 times and compute the average number of rounds that the gamble plays until they lose all the money.

In [ ]:
# Initialise a variable to hold the sum of the number of rounds
sum_of_rounds = 0

# Call the gamble function 100 times
for i in range(10000):
    sum_of_rounds += gamble(1)

# Calculate the mean number of rounds
mean_rounds = sum_of_rounds / 10000

# Output the average number of rounds
print(f"Average number of rounds for 10000 trials: {mean_rounds}")

Average number of rounds for 100 trials: 20064.8092


Run the experiments above a few times and look at the outcomes. Make a guess about the expected number of rounds that a gambler starting with money 1 can play until they lose all their money.

# Exercise 2
In this exercise, we implement the PageRank algorithm from the lectures. We assume that we have $N$ web pages and a list `links` telling us the structure of the web. The list `links` has exactly `N` entries, and each entry `links[i]` for every $0 \leq i \leq N-1$ is a list of integers (all in the range $[0, N-1]$). The presence of an integer $j$ in `links[i]` means that there is a link from webpage $i$ to webpage $j$. The parameter $d$ for PageRank is assumed to be $0.85$ in this example, and we assume that each list `link[i]` has no duplicates for every $i$.

Let's start with a 3-page web in this example, but of course you can change this web to some bigger ones.

In [24]:
N = 3
links = [[1,2],
         [2],
         []]
d = 0.85

Firstly, let's construct the transition probability matrix `P`.

In [25]:
P = np.full((N,N), 0.0, dtype=float)  # Note that it is 0.0 rather 0 to make it an array of floats rather integers.
# Iterate through each row
for i in range(N):
    # Iterate through each element in this row
    for j in range(N):
        # If this page has links, the probability of jumping to one of those linked pages is ((1-d)/N) + (d/|Lw|)
        if (len(links[i]) > 0) and (j in links[i]):
            # Assign the probability of this new page in the transition matrix P
            P[i][j] = ((1-d)/N) + (d/len(links[i]))
        elif (len(links[i]) > 0) and (j not in links[i]):
            # Assign the probability of this new page being a random other page
            P[i][j] = (1-d)/N
        else:
            # This page has no links, so could jump to any other page
            P[i][j] = 1/N


print(P)

[[0.05       0.475      0.475     ]
 [0.05       0.05       0.9       ]
 [0.33333333 0.33333333 0.33333333]]


Now we randomly generate an initial distribution `l` and multipy it with `P` iteratively until the change in every entry is smaller than $10^{-5}$. Print the distribution in each round. Run the code multiple times and see how all initial values converge to the same distribution.

In [46]:
# Set the output float precision
np.set_printoptions(precision=8)

# Generate a random initial state vector
l = np.random.rand(N)
l = l / sum(l)
epsilon = 1e-5
i = 100
lP = l @ P
lP_next = 0
stop_iterating = False

while not stop_iterating:
    # Mutliply l by P
    lP_next = lP @ P

    # Output lP
    print(lP)

    # Check if the change is within the tolerance
    if abs(lP_next[0] - lP[0]) < epsilon:
        stop_iterating = True
    else:
        lP = lP_next

[0.12359706 0.22377683 0.65262611]
[0.23491073 0.28743948 0.47764979]
[0.18533411 0.28517117 0.52949473]
[0.20002351 0.2787905  0.52118599]
[0.19766936 0.28267935 0.51965128]
[0.19723453 0.28124401 0.52152146]
[0.19776441 0.28158909 0.5206465 ]
[0.19751651 0.28156638 0.52091711]
[0.19759318 0.2815377  0.52086912]
[0.19757958 0.28155669 0.52086373]


Apart from iteration, another way to compute the stationary distribution is directly solving the equations $$(P^\intercal - I) x = 0 \quad\quad x_0 + x_1 + \cdots + x_{N-1} = 1.$$
There is a function `np.linalg.solve(Q, b)` from `numpy` that takes in a **square matrix** `Q` and a vector `b` and solves the equation `Qx = b`. We'd like to use this function to solve our equations above. As mentioned in the lecture, the square matrix $P^\intercal - I$ has rank $N-1$ rather than $N$, so we can replace any row of it (say, the first row) with $[1, 1, \cdots, 1]$ to encode the second equation above. Complete the code below.

In [ ]:
Q = np.matrix.transpose(P) - np.identity(N)
b = np.full((N,1), 0.0)
# write your code here (HINT: modify Q[0] and b[0] to encode the equation on the RHS above)
np.linalg.solve(Q, b)